##Microscope examples

Imports

In [1]:
import sys
sys.path.append('..')

import numpy as np
from PIL import Image
from src.elements import *
from src.systems import *
from src.utilities import *
from tqdm import tqdm

###Microscope model from paper

In [ ]:
nikon_microscope = OpticalSystem(color=True)

    # 60×, 1.4 NA Objective from Nikon Patent

nikon_obj_coverglass = FreeSpace(0.17, n=1.52210)
nikon_obj_oil = FreeSpace(0.13, n=1.51299)

nikon_obj_lens_1 = Doublet(
    1e6, -2.243, -3.827, 0.75, 3.85,
    refractive_index(0.5876, 'S_NSL3'),
    refractive_index(0.5876, 'S_LAH79')
)

nikon_obj_space_1 = FreeSpace(0.1)

nikon_obj_lens_2 = ThickLens(
    -23.274, -8.761, 5,
    refractive_index(0.5876, 'GFK68')
)

nikon_obj_space_2 = FreeSpace(0.15)

nikon_obj_lens_3 = Doublet(
    -38.045, 16.326, -15.9, 1, 11,
    refractive_index(0.5876, 'E_F2'),
    refractive_index(0.5876, 'GFK70')
)

nikon_obj_space_3 = FreeSpace(0.15)

nikon_obj_lens_4 = Doublet(
    331.735, 17, -17.778, 1, 10.4,
    refractive_index(0.5876, 'N_KZFS8'),
    refractive_index(0.5876, 'LITHO_CAF2')
)

nikon_obj_space_4 = FreeSpace(0.15)

nikon_obj_lens_5 = Doublet(
    34.108, 16.2, -103.612, 1, 5.6,
    refractive_index(0.5876, 'N_KZFS5'),
    refractive_index(0.5876, 'LITHO_CAF2')
)

nikon_obj_space_5 = FreeSpace(1)

nikon_obj_lens_6 = Doublet(
    17, -129.879, 21.365, 4.1, 1,
    refractive_index(0.5876, 'LITHO_CAF2'),
    refractive_index(0.5876, 'N_KZFS5')
)

nikon_obj_space_6 = FreeSpace(0.15)

nikon_obj_lens_7 = Doublet(
    9.002, -48.082, 5.9, 6.1, 2.65,
    refractive_index(0.5876, 'J_PSK03'),
    refractive_index(0.5876, 'J_LASF015')
)

nikon_obj_space_7 = FreeSpace(4.45)

nikon_obj_lens_8 = Doublet(
    -6.584, 20.8, -11.342, 1, 3.4,
    refractive_index(0.5876, 'S_LAH66'),
    refractive_index(0.5876, 'J_SF03')
)

nikon_obj_space_8 = FreeSpace(4.9719)

nikon_microscope.add_element(nikon_obj_coverglass)
nikon_microscope.add_element(nikon_obj_oil)
nikon_microscope.add_element(nikon_obj_lens_1, material=['S_NSL3', 'S_LAH79'])
nikon_microscope.add_element(nikon_obj_space_1)
nikon_microscope.add_element(nikon_obj_lens_2, material=['GFK68'])
nikon_microscope.add_element(nikon_obj_space_2)
nikon_microscope.add_element(nikon_obj_lens_3, material=['E_F2', 'GFK70'])
nikon_microscope.add_element(nikon_obj_space_3)
nikon_microscope.add_element(nikon_obj_lens_4, material=['N_KZFS8', 'LITHO_CAF2'])
nikon_microscope.add_element(nikon_obj_space_4)
nikon_microscope.add_element(nikon_obj_lens_5, material=['N_KZFS5', 'LITHO_CAF2'])
nikon_microscope.add_element(nikon_obj_space_5)
nikon_microscope.add_element(nikon_obj_lens_6, material=['LITHO_CAF2', 'N_KZFS5'])
nikon_microscope.add_element(nikon_obj_space_6)
nikon_microscope.add_element(nikon_obj_lens_7, material=['J_PSK03', 'J_LASF015'])
nikon_microscope.add_element(nikon_obj_space_7)
nikon_microscope.add_element(nikon_obj_lens_8, material=['S_LAH66', 'J_SF03'])
nikon_microscope.add_element(nikon_obj_space_8)

# Tube Lens from Nikon Patent

nikon_tube_lens_1 = Doublet(
    75.043, -75.043, 1600.58, 5.1, 2,
    refractive_index(0.5876, 'E_SK10'),
    refractive_index(0.5876, 'J_LAF7')
)

nikon_tube_lens_space_1 = FreeSpace(7.5)

nikon_tube_lens_2 = Doublet(
    50.256, -84.541, 36.911, 5.1, 1.8,
    refractive_index(0.5876, 'BASF6'),
    refractive_index(0.5876, 'KZFH1')
)

# Final distance from the tube lens
d_img = 138  # Distance from the tube lens to the image plane in mm
nikon_tube_lens_space_2 = FreeSpace(d_img)


# Add tube lens to the Nikon microscope system

nikon_microscope.add_element(nikon_tube_lens_1, material=['E_SK10', 'J_LAF7'])
nikon_microscope.add_element(nikon_tube_lens_space_1)
nikon_microscope.add_element(nikon_tube_lens_2, material=['BASF6', 'KZFH1'])
nikon_microscope.add_element(nikon_tube_lens_space_2)

# Image formation

usaf_1951_path = '../assets/usaf 1951.jpg'
usaf_1951_image = Image.open(usaf_1951_path).convert('L')
usaf_1951_array = np.array(usaf_1951_image)
usaf_1951_object = Object(usaf_1951_array, distance=0.01, height=0.2)   
pupil_radius = 0.2085

n_rays_per_pixel = 7

nikon_camera = Sensor(sensor_preset='Microscopy_SCMOS')
nikon_microscope.add_element(nikon_camera)

nikon_output_image_array = nikon_microscope.image_object(usaf_1951_object, pupil_radius=pupil_radius, n_rays_per_pixel=n_rays_per_pixel, interpolation=False)
nikon_output_image = Image.fromarray(nikon_output_image_array)
#Save the output image
nikon_output_image.save('../assets/nikon_microscope_output.png')
print("Nikon microscope output image saved as 'nikon_microscope_output.png' in the assets folder.")